# W4.2 — Kiểm chứng training framework

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** training_demo.json; toy regression train/validation riêng.

Toy y=2x+1 dùng để kiểm tra cơ chế tối ưu, không báo kết quả Cora.
Early stopping dùng strict improvement vượt min_delta; dừng sau patience epoch liên tiếp không cải thiện.
Best checkpoint được restore trước khi trả model.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/runtime.ipynb').as_posix() + '"')
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/training.ipynb').as_posix() + '"')

In [ ]:
config = read_json(PROJECT_ROOT/'configs/training_demo.json')
print(config)

{'dataset': 'toy', 'model': 'linear_regression', 'task': 'regression', 'seed': 42, 'device': 'cpu', 'learning_rate': 0.05, 'weight_decay': 0.0, 'max_epochs': 150, 'patience': 12, 'monitor': 'validation_mse', 'mode': 'min', 'min_delta': 1e-06}


In [ ]:
checks = {}
for mode,seq,want in [('min',[3.,2.,1.95,1.96],1),('max',[1.,2.,2.05,2.04],1)]:
    stopper = EarlyStopping(mode,patience=2,min_delta=.1)
    flags = [stopper.update(s,i) for i,s in enumerate(seq)]
    assert stopper.best_epoch == want and flags[-1][1] and not flags[-2][1]
    checks['early_stopping_'+mode] = {'status':'passed','best_epoch':stopper.best_epoch,'flags':flags}
checks['invalid_mode'] = expect_error(lambda:EarlyStopping('bad'))
checks['nan_validation'] = expect_error(lambda:EarlyStopping().update(float('nan'),0))
seed_everything(config['seed'])
device = torch.device(config['device'])
train_x = torch.linspace(-1,1,32,device=device).reshape(-1,1)
val_x = torch.linspace(-.95,.95,17,device=device).reshape(-1,1)
train_input = (train_x,2*train_x+1)
validation_input = (val_x,2*val_x+1)
model = torch.nn.Linear(1,1).to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=config['learning_rate'],weight_decay=config['weight_decay'])

def train_loss(model,batch):
    assert model.training and torch.is_grad_enabled()
    return torch.nn.functional.mse_loss(model(batch[0]),batch[1])

def validate(model,batch):
    assert not model.training and not torch.is_grad_enabled()
    return {'validation_mse':float(torch.nn.functional.mse_loss(model(batch[0]),batch[1]))}

run = new_run(PROJECT_ROOT,'w4_training',config)
outcome = fit(model,optimizer,train_input,validation_input,train_loss,validate,config,run)
state = torch.load(outcome['checkpoint'],map_location=device,weights_only=True)
assert outcome['best_epoch'] == state['epoch']
for key,value in model.state_dict().items(): torch.testing.assert_close(value,state['model_state'][key])
with torch.no_grad(): restored = validate(model,validation_input)['validation_mse']
assert abs(restored-outcome['best_score']) < 1e-10
assert outcome['history'][-1]['train_loss'] < outcome['history'][0]['train_loss']
checks['optimizer_gradient_loss_checkpoint'] = {'status':'passed','best_epoch':outcome['best_epoch'],'restored_validation_mse':restored,'epochs':len(outcome['history'])}
# Chứng minh fit restore epoch tốt nhất, không chỉ tải state cuối: score điều khiển 3,1,2,3.
probe = torch.nn.Linear(1,1).to(device)
probe_optimizer = torch.optim.SGD(probe.parameters(),lr=.01)
sequence = iter([3.,1.,2.,3.])
probe_config = dict(config,max_epochs=10,patience=2,min_delta=0.)
probe_run = new_run(PROJECT_ROOT,'w4_checkpoint_probe',probe_config)
probe_states = []
def controlled_validation(model,batch):
    assert not model.training and not torch.is_grad_enabled()
    probe_states.append({k:v.detach().clone() for k,v in model.state_dict().items()})
    return {'validation_mse':next(sequence)}
probe_result = fit(probe,probe_optimizer,train_input,validation_input,train_loss,controlled_validation,probe_config,probe_run)
assert probe_result['best_epoch'] == 1 and len(probe_result['history']) == 4
for key,value in probe.state_dict().items(): torch.testing.assert_close(value,probe_states[1][key])
assert any(not torch.equal(probe.state_dict()[key],probe_states[-1][key]) for key in probe.state_dict())
checks['restore_best_not_last'] = {'status':'passed','best_epoch':1,'last_epoch':3,'probe_run_id':probe_run.name}
result = publish_result(PROJECT_ROOT,run,'results/week04/training_checks.json',
    {'status':'passed','checks':checks,'validation_metric':'validation_mse','validation_value':restored},config,4,'W4.2','linear_regression','toy')
print(checks); print('run:',run.name)

{'early_stopping_min': {'status': 'passed', 'best_epoch': 1, 'flags': [(True, False), (True, False), (False, False), (False, True)]}, 'early_stopping_max': {'status': 'passed', 'best_epoch': 1, 'flags': [(True, False), (True, False), (False, False), (False, True)]}, 'invalid_mode': {'status': 'passed', 'observed': 'mode/patience/min_delta không hợp lệ'}, 'nan_validation': {'status': 'passed', 'observed': 'Validation metric không hữu hạn'}, 'optimizer_gradient_loss_checkpoint': {'status': 'passed', 'best_epoch': 31, 'restored_validation_mse': 0.0003531498368829489, 'epochs': 44}, 'restore_best_not_last': {'status': 'passed', 'best_epoch': 1, 'last_epoch': 3, 'probe_run_id': '20260928T070159_w4_checkpoint_probe_c53bcea3'}}
run: 20260928T070158_w4_training_dc7d5249


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. min_delta tác động thế nào tới best epoch?
2. eval() và no_grad() khác nhau ở đâu?
3. Vì sao cần chứng minh best checkpoint khác last checkpoint?